# Exemple d'utilisation de gridmarthe [FR]

Ce tutoriel est repris de la documentation de gridmarthe et adapté pour la formation.
Documentation complète : https://gridmarthe.readthedocs.io/en/latest/

Le package gridmarthe est conçu pour faciliter la lecture/l'écriture de grille au format Marthe.
Ce notebook permet d'explorer quelques fonctionnalités de base du package et de montrer un exemple
de traitement des grilles.

## Avant-propos - xarray

La librairie gridmarthe s'appuie sur xarray (https://docs.xarray.dev/en/stable/index.html)
qui permet de manipuler des données en tableau multi-dimensionnel avec des métadonnées,
et gestion des coordonnées, du temps. Elle est donc particulièrement adaptée à
l'exploitation de grilles spatio-temporelles telles que celles de Marthe.

La librairie xarray permet également de lire/écrire des données dans différents
formats standard, et notamment le format netcdf (https://www.unidata.ucar.edu/software/netcdf).


![xarray](assets/Dataset_xarray.png)

In [ ]:
import xarray as xr

## Gridmarthe - Fonctionnalités de base

Les grilles contiennent certaines métadonnées, dont l'index du pas de temps
mais pas les dates en elles-même. Aussi, il est utile de fournir à la fonction de lecture
soit une série de date (à construire manuellement) soit un fichier pastp pour que les dates soient
lues automatiquement.

En complément, plusieurs arguments peuvent être utiles à la lecture (retrait des valeurs nulles, transformation des xy, ajout de l'indicateur de maillage ou des numéros de colonnes/lignes, etc.).

Pour évaluer les options possibles, on peut afficher l'aide de la fonction.

La documentation complète est accessible en ligne : https://gridmarthe.readthedocs.io/

In [ ]:
# import du package
import gridmarthe as gm

In [ ]:
help(gm.load_marthe_grid)

### Chargement d'une grille

In [ ]:
# lecture d'une grille marthe et stockage dans un Dataset (librairie xarray)
ds = gm.load_marthe_grid(
    './data/chasim_hallue.out', 'CHARGE',
    fpastp='./data/hallue.pastp',
    drop_nan=True # si la valeur des nans n'est pas 9999 (ex. permh, NaNs = 0), on peut spécifier la valeur
)

In [ ]:
# affichage des valeurs et attributs
ds

### Selection de pas de temps

In [ ]:
# pour sélectionner à partir des coordonnées (ATTENTION, on parle ici des coord
# au sens de l'objet Dataset, pas des valeurs de x,y : ds.coords, soit ici time et zone)
ds.sel(time=slice('2012-07-31'))
ds.isel(time=-1)  # sélection du dernier pas de temps

La grille est chargée dans un objet xarray 2D, de dimension (time, zone).
La dimension spatiale est donc 1D, contenue dans zone. Chaque zone possède un couple de coordonnées xy, stocké en variable (et non en dimension).
Ce format permet d'effecture de nombreuses opérations de manière plus efficaces qu'en 2 ou 3D + time.
Il suffit d'assigner les coordonnées (transformation en 2 ou 3D + temps) avant les fonctionnalités graphiques.

In [ ]:
# on ajoute les coordonées
ds_3d = gm.assign_coords(ds)
# a noter que les fonctions sont aussi accessibles par méthodes/attributs:
ds_3d

In [ ]:
# on peut désormais `sel` par x,y voire z si modèle multicouche.
ds_3d.sel(x=599.2, method='nearest')

In [ ]:
# on plot à un instant t
ds_3d.isel(time=-1)['charge'].plot.pcolormesh(x='x',y='y')

In [ ]:
# ou la chronique dans une maille
ds.isel(zone=255)['charge'].plot()

Pour les gigognes, une fonction de plot spécifique est proposée (`gm.plot_nested_grids()`).

Pour les modèles avec plusieurs couches, on peut récupérer les couches de surfaces~: `gm.get_min_layer(ds, [subset_layers=[3,5]])`

ou même afficher ce résultat~: `gm.plot_outcrop()`


Les fonctions natives de xarray, numpy permettent d'ores-et-déjà d'accéder à certaines fonctionnalités de Operasem (substitution de valeurs, transformation, opérations entre deux grilles, etc.)

## Modèle multicouches

In [ ]:
ds2 = gm.load_marthe_grid('data/NPC_Amorse.charg', drop_nan=True)
display(ds2)

Par défaut, les couches ne sont pas des coordonnées, mais une varible.
On ne peut donc pas `sel` la variable `z`.

In [ ]:
ds_z_6 = ds2.where(ds2['z']==6, drop=True)
display(ds_z_6)

In [ ]:
# ou alors, on met z en coordonnées
ds2_3d = gm.assign_coords(ds2)
display(ds2_3d)

In [ ]:
display(ds2_3d.sel(z=6))

In [ ]:
ds2_3d.sel(z=6)['charge'].plot.pcolormesh(vmin=0., col='time')


## Example rivieres

In [ ]:
### Exemple avec un autre type de grille
ds3 = gm.load_marthe_grid('data/Somme_2025_reg.aff_r', xyfactor=1e3)
ds3_3d = gm.assign_coords(ds3)
display(ds3)

In [ ]:
ds3_3d.isel(time=0)['afflu_rivi'].plot.pcolormesh()

In [ ]:
tmp = ds3_3d.isel(time=0)['afflu_rivi']
tmp.where(tmp > 0).plot.pcolormesh()

## Visu QGIS

In [ ]:
# export shape
gdf = gm.to_geodataframe(ds3.where(ds3['afflu_rivi']>0, drop=True))

In [ ]:
gdf.to_file('riv.gpkg')

In [ ]:
# export netcdf
ds3_3d.where(ds3_3d['afflu_rivi']>0, drop=True).to_netcdf('riv.nc')

## Exemple de traitements

In [ ]:
import pandas as pd

In [ ]:
# compute diff between grids
ds0 = ds.sel(time='1995-11-01')
ds1 = ds.sel(time='2001-02-01')

ds0['charge'] - ds1['charge']


In [ ]:
# compute anomaly
anom = ds.groupby("time.month") - ds.groupby("time.month").median(dim="time")
ds_anom = ds.copy()
ds_anom['anom'] = anom['charge']


In [ ]:
# plot colormesh
da_plot = gm.assign_coords(ds_anom).sel(time='2001-02-01')['anom']
da_plot.plot.pcolormesh()

In [ ]:
da_plot.plot.contour()

In [ ]:
da_plot.plot.contourf()

In [ ]:
# compute seasonal mean
# 1st: mean head over each season...
# ds_season = ds.copy().sel(time=slice('1997-01-01', None))
ds_season = ds_anom.resample(time="QS-DEC").median(dim='time') # ou .agg({"time": 'mean'})

In [ ]:
# ... then aggregate all seasons
ds_season_agg = ds_season.groupby("time.season").mean(dim="time")

In [ ]:
ds_season_agg
# mind the new dimension 'season'

In [ ]:
# on passe en 2d pour le plot
ds_season_agg.update({
    v: ds_season_agg[v].isel(season=0)  # remove season dim for coordinates, appears in computation
    for v in ['x', 'y', 'dx', 'dy']
})
ds_season_agg


In [ ]:
ds_season_agg_2d = gm.assign_coords(ds_season_agg)

In [ ]:
# plot this
import numpy as np
from matplotlib import pyplot as plt, colors

var = 'anom'
minval, maxval = ds_season_agg_2d[var].min(skipna=True).values, ds_season_agg_2d[var].max(skipna=True).values
fig, axes = plt.subplots(nrows=2, ncols=2) #figsize=(14, 12))
axes = axes.ravel()
for i, season in enumerate(("DJF", "MAM", "JJA", "SON")):
    ds_season_agg_2d[var].sel(season=season).plot.pcolormesh(
            # x="x", y="lat",
            ax=axes[i],
            # cmap="Spectral",
            # vmin=minval, vmax=maxval,
            norm=colors.CenteredNorm(halfrange=np.max(np.abs([minval, maxval]))),
            add_colorbar=True,
            extend="both",
        )
plt.tight_layout()

In [ ]:
# identifier un percentile
ds_5perc = ds.copy()
ds_5perc['q5'] = ds_5perc['charge'].quantile(0.05, dim='time')
# ds_5perc['q5'] = ds_5perc.groupby('time.season').quantile(0.05, dim='time')
ds_5perc

In [ ]:
ds_5perc['tresh'] = ds_5perc['charge'] < ds_5perc['q5']
# ds_5perc['tresh'].sum(dim='time')/len(ds_5perc['time'])*100
ds_5perc